# Notebook 2 — preprocessing.ipynb

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

C1: Fix the Dtypes

In [3]:
df = pd.read_csv('../data/citibike_weather_daily.csv')
df.dtypes

ride_date               str
num_rides             int64
avg_duration_min    float64
temp_f              float64
max_temp_f          float64
min_temp_f          float64
wind_speed_knots    float64
precip_in           float64
day_of_week             str
month                 int64
dtype: object

In [4]:
df['ride_date'] = pd.to_datetime(df['ride_date'])
df['ride_date'] 

0      2013-07-01
1      2013-07-02
2      2013-07-03
3      2013-07-04
4      2013-07-05
          ...    
1605   2018-05-27
1606   2018-05-28
1607   2018-05-29
1608   2018-05-30
1609   2018-05-31
Name: ride_date, Length: 1610, dtype: datetime64[us]

In [5]:
non_numeric_mask = pd.to_numeric(df['wind_speed_knots'], errors='coerce').isna() & df['wind_speed_knots'].notna()
print(f"Rows that failed numeric conversion: {non_numeric_mask.sum()}")

df['wind_speed_knots'] = pd.to_numeric(df['wind_speed_knots'], errors='coerce')
df.dtypes

Rows that failed numeric conversion: 0


ride_date           datetime64[us]
num_rides                    int64
avg_duration_min           float64
temp_f                     float64
max_temp_f                 float64
min_temp_f                 float64
wind_speed_knots           float64
precip_in                  float64
day_of_week                    str
month                        int64
dtype: object

C2: Handle the Coded Missing Values

In [7]:
sentinel_map = {
    'temp_f': 9999.9,
    'max_temp_f': 9999.9,
    'min_temp_f': 9999.9,
    'wind_speed_knots': 9999.9,
    'precip_in': 9999.9,

}

for col, sentinel in sentinel_map.items():
    df[col] = df[col].replace(sentinel, np.nan)

df.isna().sum()

ride_date           0
num_rides           0
avg_duration_min    0
temp_f              0
max_temp_f          0
min_temp_f          0
wind_speed_knots    0
precip_in           0
day_of_week         0
month               0
dtype: int64

C3: Encode Day of Week

In [29]:
df['day_of_week_original'] = df['ride_date'].dt.day_name()

df = pd.get_dummies(df, columns=['day_of_week_original'], drop_first=True)
df.head()


,ride_date,num_rides,avg_duration_min,temp_f,max_temp_f,min_temp_f,wind_speed_knots,precip_in,month,day_of_week_original_Monday,day_of_week_original_Saturday,day_of_week_original_Sunday,day_of_week_original_Thursday,day_of_week_original_Tuesday,day_of_week_original_Wednesday,day_of_week_Monday,day_of_week_Saturday,day_of_week_Sunday,day_of_week_Thursday,day_of_week_Tuesday,day_of_week_Wednesday,day_of_week_original_Monday,day_of_week_original_Saturday,day_of_week_original_Sunday,day_of_week_original_Thursday,day_of_week_original_Tuesday,day_of_week_original_Wednesday,day_of_week_original_Monday,day_of_week_original_Saturday,day_of_week_original_Sunday,day_of_week_original_Thursday,day_of_week_original_Tuesday,day_of_week_original_Wednesday,day_of_week_original_Monday,day_of_week_original_Saturday,day_of_week_original_Sunday,day_of_week_original_Thursday,day_of_week_original_Tuesday,day_of_week_original_Wednesday
0,2013-07-01,16650,16.309988,74.8,78.1,73.4,7.8,0.00,7,True,False,False,False,False,False,True,False,False,False,False,False,True,False,False,False,False,False,True,False,False,False,False,False,True,False,False,False,False,False
1,2013-07-02,22745,15.968826,76.1,82.9,73.0,8.0,0.73,7,False,False,False,False,True,False,False,False,False,False,True,False,False,False,False,False,True,False,False,False,False,False,True,False,False,False,False,False,True,False
2,2013-07-03,21864,16.238808,78.5,84.9,73.9,8.8,0.06,7,False,False,False,False,False,True,False,False,False,False,False,True,False,False,False,False,False,True,False,False,False,False,False,True,False,False,False,False,False,True
3,2013-07-04,22326,21.218474,82.0,91.0,73.9,8.6,0.96,7,False,False,False,True,False,False,False,False,False,True,False,False,False,False,False,True,False,False,False,False,False,True,False,False,False,False,False,True,False,False
4,2013-07-05,21842,18.040443,84.4,93.0,75.9,9.0,0.00,7,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False


C4: Build a Trend Feature

In [30]:
df['year'] = df['ride_date'].dt.year
df['days_since_launch'] = (df['ride_date'] - df['ride_date'].min()).dt.days
df[['ride_date', 'year', 'days_since_launch']].head()

,ride_date,year,days_since_launch
0,2013-07-01,2013,0
1,2013-07-02,2013,1
2,2013-07-03,2013,2
3,2013-07-04,2013,3
4,2013-07-05,2013,4


C5 (Stretch): Engineer Smarter Features

C6: Save the Clean Dataset

In [38]:
df = df.drop(columns=['day_of_week_original'], errors='ignore')

df.to_csv('../data/citibike_weather_daily_clean.csv', index=False)
print("Saved cleaned dataset:", df.shape)
df.head()

Saved cleaned dataset: (1610, 41)


,ride_date,num_rides,avg_duration_min,temp_f,max_temp_f,min_temp_f,wind_speed_knots,precip_in,month,day_of_week_original_Monday,day_of_week_original_Saturday,day_of_week_original_Sunday,day_of_week_original_Thursday,day_of_week_original_Tuesday,day_of_week_original_Wednesday,day_of_week_Monday,day_of_week_Saturday,day_of_week_Sunday,day_of_week_Thursday,day_of_week_Tuesday,day_of_week_Wednesday,day_of_week_original_Monday,day_of_week_original_Saturday,day_of_week_original_Sunday,day_of_week_original_Thursday,day_of_week_original_Tuesday,day_of_week_original_Wednesday,day_of_week_original_Monday,day_of_week_original_Saturday,day_of_week_original_Sunday,day_of_week_original_Thursday,day_of_week_original_Tuesday,day_of_week_original_Wednesday,day_of_week_original_Monday,day_of_week_original_Saturday,day_of_week_original_Sunday,day_of_week_original_Thursday,day_of_week_original_Tuesday,day_of_week_original_Wednesday,year,days_since_launch
0,2013-07-01,16650,16.309988,74.8,78.1,73.4,7.8,0.00,7,True,False,False,False,False,False,True,False,False,False,False,False,True,False,False,False,False,False,True,False,False,False,False,False,True,False,False,False,False,False,2013,0
1,2013-07-02,22745,15.968826,76.1,82.9,73.0,8.0,0.73,7,False,False,False,False,True,False,False,False,False,False,True,False,False,False,False,False,True,False,False,False,False,False,True,False,False,False,False,False,True,False,2013,1
2,2013-07-03,21864,16.238808,78.5,84.9,73.9,8.8,0.06,7,False,False,False,False,False,True,False,False,False,False,False,True,False,False,False,False,False,True,False,False,False,False,False,True,False,False,False,False,False,True,2013,2
3,2013-07-04,22326,21.218474,82.0,91.0,73.9,8.6,0.96,7,False,False,False,True,False,False,False,False,False,True,False,False,False,False,False,True,False,False,False,False,False,True,False,False,False,False,False,True,False,False,2013,3
4,2013-07-05,21842,18.040443,84.4,93.0,75.9,9.0,0.00,7,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,2013,4
